In [5]:
# importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os

from tensorflow.keras.models import load_model

# ignore warning
import warnings
warnings.filterwarnings("ignore")

In [7]:
DIR_PATH = os.getcwd()
DIR_PATH

'd:\\Aastha\\MLEngineering\\ml_practice\\DeepLearning\\ANN_Project'

In [8]:
# Loading all the pickle file
with open(f"{DIR_PATH}/gender_mapping.pkl", "rb") as f:
    gender_mapping = pickle.load(f)

with open(f"{DIR_PATH}/onehot_geography.pkl", "rb") as f:
    onehot_geography = pickle.load(f)

with open(f"{DIR_PATH}/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)


# loading model
model = load_model(f"{DIR_PATH}/model.h5")

In [9]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [10]:
# Tranforming the Geography column for the input data
geo_encoder = onehot_geography.transform([[input_data['Geography']]]).toarray()

geo_encoder_df = pd.DataFrame(geo_encoder, columns=onehot_geography.get_feature_names_out())


geo_encoder_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [11]:
# converting dictionary into a dataframe
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [12]:
# Transforming Gender column
input_df['Gender'] = input_df['Gender'].map(gender_mapping)
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,0,40,3,60000,2,1,1,50000


In [13]:
input_df = pd.concat([input_df.drop('Geography', axis=1), geo_encoder_df], axis=1)

In [14]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,0,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [15]:
# scaling the input data
scaled_df = scaler.transform(input_df)
scaled_df

array([[-0.53598516, -0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [16]:
# Making predictions
predictions = model.predict(scaled_df)

prediction_probability = predictions[0][0]
prediction_probability

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step


np.float32(0.031203054)

In [17]:
if prediction_probability > 0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is not likely to churn.")

The customer is not likely to churn.
